# 🧪 Sesión 14 · Hashing Seguro de Contraseñas (Bcrypt / Argon2)

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Por qué nunca se guarda una contraseña tal cual?

---

## 🖥️ Así se verá el resultado final

```http
POST /api/registro  {"usuario":"ana","password":"secreta"}
HTTP/1.1 201 Created
{"usuario":"ana"}

POST /api/login  {"usuario":"ana","password":"secreta"}
HTTP/1.1 200 OK
{"usuario":"ana","autenticado":true}

POST /api/login  {"usuario":"ana","password":"incorrecta"}
HTTP/1.1 401 Unauthorized

La respuesta NUNCA contiene "password" ni el hash.
```

_El hash se guarda en el servidor; la contraseña original no se almacena en ningún lado._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Guardar contraseñas con bcrypt y salt.
- Verificar una contraseña sin descifrarla.
- Comprobar que la API nunca devuelve el hash ni la contraseña.

## ✅ Criterios de aceptación

El verificador ejecutará los casos HTTP y comprobará los archivos del contrato. Todo debe quedar en verde:

- [ ] El registro responde sin filtrar la contraseña → HTTP 201
- [ ] El login correcto autentica → HTTP 200
- [ ] La contraseña incorrecta se rechaza → HTTP 401
- [ ] Se usa un algoritmo de hashing para contraseñas

## 🧩 El reto

Implementa registro y login con hashing seguro. La contraseña nunca debe viajar de vuelta al cliente.

**Requisitos del contrato**

- POST /api/registro recibe usuario y password, guarda solo el hash y responde 201.
- La respuesta de registro no incluye password ni hash.
- POST /api/login compara la contraseña con el hash y responde 200 o 401.
- Debes usar bcrypt (o passlib) para el hash.
- Un usuario repetido no debe sobrescribir la contraseña existente.

> **Pista conceptual:** No compares hashes con `==` si puedes usar la función de verificación del algoritmo. Bcrypt ya incluye el salt dentro del hash.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion14.md` explicando por qué no se puede «deshashear».


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:

%%writefile app.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import bcrypt

app = FastAPI(title="Registro y Login Seguro")

# Diccionario para guardar los usuarios y sus hashes
usuarios = {}

# Modelo para recibir los datos
class UsuarioEntrada(BaseModel):
    usuario: str = Field(min_length=1)
    password: str = Field(min_length=1)


# Ruta para registrar un usuario
@app.post("/api/registro", status_code=201)
def registrar_usuario(datos: UsuarioEntrada):

    # Comprobar si el usuario ya existe
    if datos.usuario in usuarios:
        raise HTTPException(
            status_code=409,
            detail="El usuario ya existe"
        )

    # Convertir la contraseña en un hash seguro
    password_hash = bcrypt.hashpw(
        datos.password.encode("utf-8"),
        bcrypt.gensalt()
    )

    # Guardar únicamente el hash
    usuarios[datos.usuario] = password_hash

    # Devolver solamente el nombre del usuario
    return {"usuario": datos.usuario}


# Ruta para iniciar sesión
@app.post("/api/login")
def iniciar_sesion(datos: UsuarioEntrada):

    # Buscar el hash del usuario
    password_hash = usuarios.get(datos.usuario)

    # Verificar que exista y comprobar la contraseña
    if password_hash is None or not bcrypt.checkpw(
        datos.password.encode("utf-8"),
        password_hash
    ):
        raise HTTPException(
            status_code=401,
            detail="Usuario o contraseña incorrectos"
        )

    return {
        "usuario": datos.usuario,
        "autenticado": True
    }


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Python 3.10 o superior, FastAPI y bcrypt, trabajando desde Google Colab.|
| **Rol** |Actuar como un desarrollador especializado en seguridad web y creación de APIs REST. |
| **Especificación** |Crear las rutas de registro y login. Guardar las contraseñas mediante bcrypt, verificar las contraseñas ingresadas y evitar que se repitan los usuarios.|
| **Output** |	Entregar el código completo de app.py, con modelos Pydantic, hashing seguro y respuestas HTTP 201, 200, 401 y 409.|

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'El registro responde sin filtrar la contraseña',
  'metodo': 'POST',
  'ruta': '/api/registro',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 201,
  'json_contiene': {'usuario': 'ana'},
  'no_contiene': ['password', 'hash']},
 {'descripcion': 'El login correcto autentica',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'secreta'},
  'estado': 200,
  'json_contiene': {'usuario': 'ana', 'autenticado': True}},
 {'descripcion': 'La contraseña incorrecta se rechaza',
  'metodo': 'POST',
  'ruta': '/api/login',
  'json': {'usuario': 'ana', 'password': 'incorrecta'},
  'estado': 401}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 3 casos · app.py
✅ 01. El registro responde sin filtrar la contraseña
✅ 02. El login correcto autentica
✅ 03. La contraseña incorrecta se rechaza
------------------------------------------------------------------------------
RESULTADO: 3/3 casos correctos. ¡Resultado logrado!


In [ ]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("app.py", regex=["bcrypt|passlib"], descripcion="Se usa un algoritmo de hashing para contraseñas"))
resumen(resultados)


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion14.md` a `auditorias/auditoria_sesion14.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue entender cómo comprobar una contraseña sin tener que guardarla directamente. Al principio no tenía muy claro cómo funcionaba el hash, pero aprendí que bcrypt permite convertir la contraseña en un código seguro y después verificarla cuando el usuario intenta iniciar sesión.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí utilizar un diccionario llamado usuarios para guardar los nombres de usuario junto con sus contraseñas convertidas en hash. También utilicé bcrypt.gensalt() para generar un salt diferente y hacer más seguro el almacenamiento de las contraseñas.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   Durante la revisión tuve que comprobar que las respuestas de la API no mostraran la contraseña ni el hash. También revisé que el login rechazara las contraseñas incorrectas con el código 401 y que el sistema no permitiera reemplazar la contraseña de un usuario que ya estaba registrado.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
